# 天気図パターン分類 - 手元で使う (v3)

VS Code でこのノートブックを開いて、上から順に実行してください。
**このリポジトリのフォルダの中だけで完結します**(GitHubにもColabにも繋ぎません)。

Colab版(`notebooks/predict.ipynb`)と**同じ関数**を呼んでいます
(`src/quicklook.py`)。片方だけ直して食い違うことがないよう、中身は1か所に
まとめてあります。

置き場所の決まり:

| | 場所 |
|---|---|
| 2023年以降の天気図(前処理後) | `data/processed/jma/` |
| 2000〜2022年の天気図(前処理後) | `data/processed/ndl/` |
| 重み | `weights/model.pt`・`weights/model_annot.pt` |
| H/L のテンプレート | `data/templates/` |

揃っているかは `python -m scripts.check_local_setup` で確かめられます。
足りないものがあれば、用意するコマンドまで出ます。

In [ ]:
# セットアップ(最初に1回だけ)
import sys
from pathlib import Path

# リポジトリのルートを import できるようにする。VS Code は
# ノートブックのある場所をカレントにすることがあるので、両方に対応する
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# **どのPythonで動いているかを必ず出す。**VS Code のカーネルは、
# ターミナルで有効化した仮想環境とは別のものが選ばれていることがある。
# その場合「ターミナルでは動くのにノートブックでは ModuleNotFoundError」
# という分かりにくい状態になる。
print(f"リポジトリ: {ROOT}")
print(f"Python    : {sys.executable}")

_missing = []
for _name in ("torch", "torchvision", "matplotlib", "cv2", "pandas", "PIL", "scipy"):
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

if _missing:
    print("\n★足りない道具: " + ", ".join(_missing))
    print("  上に出ている Python が、いつも使っている仮想環境と違っていませんか。")
    print("  違う場合: VS Code 右上の「カーネルの選択」で wpcvenv を選ぶ")
    print("  同じ場合: そのPythonに入れる ->")
    print(f'    & "{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"')
else:
    %matplotlib inline

    from src.quicklook import annotation_available, classify_and_show

    ok, missing = annotation_available()
    print("注釈方式: " + ("使えます" if ok else f"使えません(足りない: {missing})"))
    print()
    print("揃っているかまとめて見るには、ターミナルで:")
    print("  python -m scripts.check_local_setup")


## 天気図を1枚分類する

In [ ]:
# ここを書き換えて実行する
#
# **どの時代の天気図でも同じ書き方でよい。**検出の設定(テンプレートの倍率と
# しきい値)は、ファイル名の日付から自動で選ばれる。2023年以降なら学習時と
# まったく同じ設定、それ以前なら古い天気図用の設定になる。
#
# パスはすべてこのリポジトリの中で完結させる。ROOT からの相対で書けば、
# ノートブックをどこから開いても、フォルダごと移動しても動く。

IMAGE = ROOT / "data" / "processed" / "jma" / "Js_2023010100.png"
# IMAGE = ROOT / "data" / "processed" / "ndl" / "JS_2000010100_page001.png"

# 表示するラベルのしきい値。None にすると校正ファイルのラベルごとの値を使う
THRESHOLD = 0.5

# 検出した枠を描き込んでから分類する(左端の絵で検出の当たり外れが見える)
USE_ANNOTATION = True

classify_and_show(IMAGE, threshold=THRESHOLD, annotate=USE_ANNOTATION)
